# 🤖 의도 파악 생성 모델 파인튜닝 (nlu-gen-v1, Colab · LoRA)

Claude 대신 **우리 서버에서 직접 돌리는 작은 LLM** 을 만듭니다. 손님 말을 키오스크 표준 문장으로 바꿔 씁니다.
분류 모델(nlu-intent-v1)과 달리 **메뉴 이름이 들어간 말**도 처리합니다.
예: (메뉴) "그 매운 닭 버거 두 개" → `맥스파이시 상하이 버거 2개 주세요`, (세트 옵션) "콜라 말고 주스로" → `코카콜라 말고 오렌지 주스로 바꿔 주세요`

- 기반 모델: `Qwen/Qwen2.5-1.5B-Instruct` (Apache-2.0, 한국어 가능) + LoRA
- 결과: `nlu-gen-v1.zip` (GGUF, 약 1GB). 모델 서버가 CPU 로 실행합니다 (llama.cpp)

| 단계 | 하는 일 | 걸리는 시간 (A100) |
|---|---|---|
| 0 | GPU 확인 · 라이브러리 설치 | 2분 |
| 1 | 데이터 올리기 (`nlu_data.zip`, 분류 모델과 같은 파일) | 1분 |
| 2 | 원본 모델 실력 보기 (기준점) | 1분 |
| 3 | LoRA 학습 | 5~10분 |
| 4 | 처음 보는 말 시험 | 1분 |
| 5 | 합치기 · GGUF 변환 · 줄이기(Q4) | 10분 |
| 6 | Drive 저장 · 내려받기 | 2~5분 |

**사용법:** `런타임 → 런타임 유형 변경 → A100 GPU` 선택 후 위에서부터 셀을 하나씩 실행(▶)하세요.
1단계에서 Drive 연결을 허용하면 PC 와 동기화된 **`다른 컴퓨터/내 PC/프로젝트1/colab`** 에서 데이터를 가져오고, 결과(약 1GB)도 거기에 저장합니다.

## 0. GPU 확인 · 라이브러리 설치

In [ ]:
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "accelerate", "gguf", "sentencepiece"], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

## 1. 데이터 올리기

In [ ]:
import os, zipfile, shutil
WORK = "/content/nlu_gen" if IN_COLAB else os.environ.get("NLU_WORK", "./nlu_gen_work")
os.makedirs(WORK, exist_ok=True)
ZIP = os.path.join(WORK, "nlu_data.zip")
# Drive 에서 데이터 폴더 찾기: PC 와 동기화된 "다른 컴퓨터/…/프로젝트1/colab" 먼저, 없으면 "내 드라이브/fundamental1"
import glob
def find_drive_dir():
    cands = glob.glob("/content/drive/Othercomputers/*/프로젝트1/colab") + ["/content/drive/MyDrive/fundamental1"]
    for d in cands:
        if os.path.exists(os.path.join(d, "nlu_data.zip")):
            return d
    return None
DRIVE_DIR = None

if IN_COLAB:
    from google.colab import drive, files
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive 연결 안 함 (결과는 브라우저로 내려받습니다):", e)
if not os.path.exists(ZIP):
    if IN_COLAB:
        DRIVE_DIR = find_drive_dir()
        if DRIVE_DIR:
            shutil.copy(os.path.join(DRIVE_DIR, "nlu_data.zip"), ZIP)
            print("Drive 에서 가져옴:", DRIVE_DIR)
        else:
            print("nlu_data.zip 을 골라 주세요")
            up = files.upload()
            shutil.move(next(iter(up)), ZIP)
    else:
        shutil.copy(os.environ["NLU_DATA_ZIP"], ZIP)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(WORK)
DATA = os.path.join(WORK, "nlu")

import csv, json, random
def read_jsonl(name):
    return [json.loads(l) for l in open(os.path.join(DATA, name), encoding="utf-8")]
def read_csv(name):
    return list(csv.DictReader(open(os.path.join(DATA, name), encoding="utf-8")))
train, val = read_jsonl("gen_train.jsonl"), read_jsonl("gen_val.jsonl")
meta = json.load(open(os.path.join(DATA, "meta.json"), encoding="utf-8"))
STEP_NAMES = meta["step_names"]
UNKNOWN = "(모름)"
# 시험: 사람이 쓴 처음 보는 말 (분류용 + 메뉴 이름이 들어간 말). NONE/빈칸 → (모름)
test = [{"step": r["step"], "text": r["text"], "rewrite": "" if r["label"] in ("NONE", "") else r["label"]}
        for r in read_csv("testset.csv") + read_csv("gen_testset.csv")]
if os.environ.get("NLU_SMOKE"):
    train, val, test = train[:64], val[:16], test[:8]
print(f"학습 {len(train)} · 검증 {len(val)} · 시험 {len(test)}")
for r in random.Random(0).sample(train, 5):
    print(" ", r)

## 2. 원본 모델 실력 보기 (기준점)
학습 전 모델에게 같은 시험을 봅니다. 학습 뒤와 비교하려는 것입니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE = os.environ.get("NLU_GEN_BASE", "Qwen/Qwen2.5-1.5B-Instruct")
SYSTEM = ("너는 패스트푸드점 음성 키오스크의 말 정리 담당이다. 손님 말을 키오스크가 알아듣는 표준 문장 하나로 바꾼다. "
          "메뉴는 매장 메뉴 이름 그대로 쓰고 수량은 숫자로 쓴다. 뜻이 분명하지 않거나 주문과 상관없으면 (모름) 이라고만 답한다.")
USER_FORMAT = "단계: {step}\n손님: {text}"

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
END = "<|im_end|>"
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=DTYPE).to(DEVICE)

def prompt_text(step, text):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": USER_FORMAT.format(step=STEP_NAMES.get(step, step), text=text)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(rows, bs=int(os.environ.get("NLU_GEN_BS", 32))):
    model.eval()
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(rows), bs):
        b = rows[i:i + bs]
        enc = tok([prompt_text(r["step"], r["text"]) for r in b], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(DEVICE)
        gen = model.generate(**enc, max_new_tokens=32, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [s.strip() for s in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return outs

def evaluate(rows, show=0):
    outs = generate(rows)
    norm = lambda s: s.replace(" ", "")
    right = 0
    errors = []
    for r, o in zip(rows, outs):
        want = r["rewrite"] or UNKNOWN
        if norm(o) == norm(want):
            right += 1
        else:
            errors.append((r["step"], r["text"], o, want))
    for e in errors[:show]:
        print(f"  [{e[0]}] {e[1]} → {e[2]} / 정답 {e[3]}")
    return right / len(rows), errors

acc0, _ = evaluate(test, show=5)
print(f"학습 전 정확도: {acc0:.1%}")

## 3. LoRA 학습

In [ ]:
import time
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

EPOCHS = int(os.environ.get("NLU_EPOCHS", 2))
BATCH = int(os.environ.get("NLU_BATCH", 16))
LR = 2e-4
MAX_STEPS = int(os.environ.get("NLU_MAX_STEPS", 0))  # 0 = 끝까지
torch.manual_seed(0)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def encode(r):
    p = tok(prompt_text(r["step"], r["text"]), add_special_tokens=False)["input_ids"]
    a = tok((r["rewrite"] or UNKNOWN) + END + "\n", add_special_tokens=False)["input_ids"]
    return p + a, [-100] * len(p) + a  # 답 부분만 배운다

def collate(b):
    pairs = [encode(r) for r in b]
    n = max(len(x) for x, _ in pairs)
    ids = torch.full((len(b), n), tok.pad_token_id)
    labels = torch.full((len(b), n), -100)
    mask = torch.zeros((len(b), n), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)] = torch.tensor(x); labels[i, :len(y)] = torch.tensor(y); mask[i, :len(x)] = 1
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}

loader = DataLoader(train, batch_size=BATCH, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
total = MAX_STEPS or EPOCHS * len(loader)
sched = get_linear_schedule_with_warmup(opt, int(total * 0.05), total)
step, t0 = 0, time.time()
model.train()
for epoch in range(EPOCHS):
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        loss = model(**b).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        step += 1
        if step % 50 == 0 or step == total:
            print(f"step {step}/{total} · loss {loss.item():.3f} · {time.time() - t0:.0f}초")
        if step >= total:
            break
    if step >= total:
        break
    vacc, _ = evaluate(val[:200])
    print(f"epoch {epoch + 1}: 검증 정확도 {vacc:.1%}")

## 4. 처음 보는 말 시험

In [ ]:
acc1, errors = evaluate(test, show=40)
print(f"\n처음 보는 말 정확도: 학습 전 {acc0:.1%} → 학습 후 {acc1:.1%}  (틀린 것 {len(errors)}개, 위에 보임)")

## 5. 합치기 · GGUF 변환 · Q4 로 줄이기
LoRA 를 원래 모델에 합친 뒤 llama.cpp 형식(GGUF)으로 바꿉니다. CPU 에서 빠르게 돌리려고 4비트(Q4_K_M)로 줄입니다.
Q4 만들기가 실패하면 8비트(Q8_0, 약 1.6GB)를 씁니다.

In [ ]:
merged_dir = os.path.join(WORK, "merged")
model = model.merge_and_unload()
model.save_pretrained(merged_dir, safe_serialization=True)
tok.save_pretrained(merged_dir)

LLAMA = os.path.join(WORK, "llama.cpp")
if not os.path.exists(LLAMA):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
OUT = os.path.join(WORK, "nlu-gen-v1")
os.makedirs(OUT, exist_ok=True)
q8 = os.path.join(WORK, "model-q8_0.gguf")
subprocess.run([sys.executable, os.path.join(LLAMA, "convert_hf_to_gguf.py"), merged_dir, "--outfile", q8,
                "--outtype", "q8_0"], check=True)
final = os.path.join(OUT, "model.gguf")
quant = "Q8_0"
if not os.environ.get("NLU_SKIP_Q4"):
    try:
        subprocess.run(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && "
                       f"cmake --build {LLAMA}/build --target llama-quantize llama-cli -j 8 > /dev/null",
                       shell=True, check=True)
        f16 = os.path.join(WORK, "model-f16.gguf")
        subprocess.run([sys.executable, os.path.join(LLAMA, "convert_hf_to_gguf.py"), merged_dir, "--outfile", f16,
                        "--outtype", "f16"], check=True)
        subprocess.run([f"{LLAMA}/build/bin/llama-quantize", f16, final, "Q4_K_M"], check=True,
                       stdout=subprocess.DEVNULL)
        quant = "Q4_K_M"
    except Exception as e:
        print("Q4 만들기 실패 → Q8 사용:", e)
if quant == "Q8_0":
    shutil.copy(q8, final)
print(f"{quant} · {os.path.getsize(final) / 1e9:.2f}GB")

## 6. 저장 · 내려받기

In [ ]:
info = {
    "model_version": "nlu-gen-v1", "base_model": BASE, "quant": quant, "system": SYSTEM, "user_format": USER_FORMAT,
    "unknown": UNKNOWN, "step_names": STEP_NAMES,
    "metrics": {"test_acc_before": round(acc0, 4), "test_acc": round(acc1, 4), "test_size": len(test)},
    "trained_at": time.strftime("%Y-%m-%d %H:%M"),
}
json.dump(info, open(os.path.join(OUT, "meta.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(info["metrics"], ensure_ascii=False))

result = os.path.join(WORK, "nlu-gen-v1.zip")
with zipfile.ZipFile(result, "w", zipfile.ZIP_STORED) as z:  # GGUF 는 이미 압축돼 있어 그냥 묶기만
    for name in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, name), f"nlu-gen-v1/{name}")
print("결과:", result, f"{os.path.getsize(result) / 1e9:.2f}GB")
if IN_COLAB:
    save_dir = DRIVE_DIR or (find_drive_dir() if os.path.isdir("/content/drive") else None)
    if save_dir:
        shutil.copy(result, os.path.join(save_dir, "nlu-gen-v1.zip"))
        print("Drive 에 저장:", os.path.join(save_dir, "nlu-gen-v1.zip"), "→ PC 의 프로젝트1/colab 폴더에 자동으로 생깁니다")
    else:
        files.download(result)

## 다 했으면
`nlu-gen-v1.zip` 이 PC 의 `프로젝트1\colab` 폴더에 생기면 (1GB 라 동기화에 몇 분 걸림) Claude 에게 "생성 모델 학습 끝났어" 라고 알려 주세요.
4단계 정확도(학습 전 → 후)는 노션 "모델 실험 기록"에 남깁니다.